# Perfilado de fuente: Toctoc

**Objetivo:** perfilar los 24 archivos mensuales de Toctoc (2023-2024) en `data/raw/toctoc/`
para detectar problemas de calidad de datos (nulos, duplicados, outliers, inconsistencias de
formato) antes de diseñar la capa `staging`. 

Este notebook es solo exploratorio: no transforma ni guarda datos intermedios. La lógica de
limpieza definitiva vive en `models/staging/`.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 20)

RAW_DIR = Path("../data/raw/toctoc")

## 0. Carga de los 24 archivos mensuales

In [2]:
files = sorted(RAW_DIR.glob("*/*.csv"))
print(f"Archivos encontrados: {len(files)}")
assert len(files) == 24, "Se esperaban 24 archivos mensuales (2023-01 a 2024-12)"

dfs = []
for f in files:
    d = pd.read_csv(f)
    d["archivo_origen"] = f.name
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
df.shape

Archivos encontrados: 24


(124374, 13)

## 1. Estructura general: columnas y tipos

In [3]:
df.dtypes

id                       str
fuente                   str
comuna                   str
region                   str
precio               float64
divisa                   str
superficie_m2        float64
fecha_publicacion        str
fecha_scraping           str
tipo_propiedad           str
tipo_operacion           str
contact_type             str
archivo_origen           str
dtype: object

In [4]:
df.head(3)

,id,fuente,comuna,region,precio,divisa,superficie_m2,fecha_publicacion,fecha_scraping,tipo_propiedad,tipo_operacion,contact_type,archivo_origen
0,TT-2023-01-000117,toctoc,Recoleta,Metropolitana,13.57,UF,40.2,2022-12-17 00:00:00.000000,2023-01-01 00:00:00.000000,DEPARTAMENTO,arriendo,NaN,2023-01.csv
1,TT-2023-01-002596,toctoc,Las Condes,Metropolitana,26.57,UF,48.8,2023-01-16 00:00:00.000000,2023-01-22 00:00:00.000000,DEPARTAMENTO,arriendo,agency,2023-01.csv
2,TT-2023-01-002558,toctoc,San Joaquín,Metropolitana,19.14,UF,60.1,2022-12-18 00:00:00.000000,2023-01-22 00:00:00.000000,DEPARTAMENTO,ARRIENDO,agency,2023-01.csv


## 2. Nulos por columna

In [5]:
nulos = df.isna().sum().to_frame("n_nulos")
nulos["pct_nulos"] = (nulos["n_nulos"] / len(df) * 100).round(2)
nulos.sort_values("n_nulos", ascending=False)

,n_nulos,pct_nulos
contact_type,3048,2.45
comuna,1328,1.07
id,0,0.00
region,0,0.00
fuente,0,0.00
precio,0,0.00
divisa,0,0.00
fecha_publicacion,0,0.00
superficie_m2,0,0.00
fecha_scraping,0,0.00


`comuna` y `contact_type` presentan nulos. `contact_type` nulo puede ser un valor legítimo
("no informado por el scraper") más que un error — queda para decidir en staging cómo tratarlo.

## 3. Duplicados

In [6]:
dup_exactos = df.duplicated().sum()
dup_id = df["id"].duplicated().sum()
print(f"Filas exactamente duplicadas (todas las columnas): {dup_exactos}")
print(f"Filas con id duplicado: {dup_id}")

Filas exactamente duplicadas (todas las columnas): 172
Filas con id duplicado: 172


In [7]:
# Si un id se repite, ¿es porque la fila completa se repite (posible reprocesamiento del scraper)
# o porque el mismo id aparece con datos distintos (colisión de clave)?
ids_dup = df[df["id"].duplicated(keep=False)].sort_values("id")
ids_dup.groupby("id").nunique().max()

fuente               1
comuna               1
region               1
precio               1
divisa               1
superficie_m2        1
fecha_publicacion    1
fecha_scraping       1
tipo_propiedad       1
tipo_operacion       1
contact_type         1
archivo_origen       1
dtype: int64

Si el máximo de valores distintos por columna entre filas con el mismo `id` es 1 (salvo
`archivo_origen`/`fecha_scraping`), los "duplicados de id" son el mismo aviso capturado más
de una vez por el scraper, no colisiones de clave.

## 4. Outliers en precio y superficie

In [8]:
df["divisa"].value_counts()

divisa
UF     68478
CLP    55896
Name: count, dtype: int64

`precio` mezcla dos unidades distintas (UF y CLP) según la columna `divisa` — no es comparable
directamente entre filas sin normalizar primero. Se describe por separado.

In [9]:
df.groupby("divisa")["precio"].describe()

,count,mean,std,min,25%,50%,75%,max
divisa,,,,,,,,
CLP,55896.0,768858.773436,989840.303705,255000.00,490000.00,605000.00,805000.00,71265000.00
UF,68478.0,21.216489,23.087390,6.89,13.65,16.76,22.43,783.14


In [10]:
print("precio <= 0:", (df["precio"] <= 0).sum())
print("superficie_m2 <= 0:", (df["superficie_m2"] <= 0).sum())
df["superficie_m2"].describe()

precio <= 0: 0
superficie_m2 <= 0: 0


count    124374.000000
mean         57.571065
std          61.050201
min           1.000000
25%          39.200000
50%          49.000000
75%          63.600000
max        1200.000000
Name: superficie_m2, dtype: float64

In [11]:
# Candidatos a outlier: valores extremos de precio dentro de cada divisa (percentil 99.5)
for divisa, g in df.groupby("divisa"):
    p995 = g["precio"].quantile(0.995)
    print(f"{divisa}: p99.5 = {p995:.2f}, max = {g['precio'].max():.2f}, "
          f"n > p99.5 = {(g['precio'] > p995).sum()}")

CLP: p99.5 = 6982875.00, max = 71265000.00, n > p99.5 = 280
UF: p99.5 = 193.18, max = 783.14, n > p99.5 = 343


## 5. Fechas: parseabilidad y consistencia

In [12]:
fecha_publicacion = pd.to_datetime(df["fecha_publicacion"], errors="coerce")
fecha_scraping = pd.to_datetime(df["fecha_scraping"], errors="coerce")

print("fecha_publicacion sin parsear:", fecha_publicacion.isna().sum())
print("fecha_scraping sin parsear:", fecha_scraping.isna().sum())
print("fecha_publicacion rango:", fecha_publicacion.min(), "→", fecha_publicacion.max())
print("fecha_scraping rango:", fecha_scraping.min(), "→", fecha_scraping.max())
print("fecha_publicacion > fecha_scraping (inconsistente):", (fecha_publicacion > fecha_scraping).sum())

fecha_publicacion sin parsear: 0
fecha_scraping sin parsear: 0
fecha_publicacion rango: 2022-05-01 00:00:00 → 2024-12-28 00:00:00
fecha_scraping rango: 2023-01-01 00:00:00 → 2024-12-29 00:00:00
fecha_publicacion > fecha_scraping (inconsistente): 0


## 6. Consistencia de valores categóricos

In [13]:
for col in ["tipo_propiedad", "tipo_operacion", "contact_type", "region"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

--- tipo_propiedad ---
tipo_propiedad
Departamento    41598
DEPARTAMENTO    41505
departamento    41271
Name: count, dtype: int64

--- tipo_operacion ---
tipo_operacion
ARRIENDO    41690
arriendo    41401
Arriendo    41283
Name: count, dtype: int64

--- contact_type ---
contact_type
agency          78880
owner_direct    42446
NaN              3048
Name: count, dtype: int64

--- region ---
region
Metropolitana    124374
Name: count, dtype: int64



`tipo_propiedad` y `tipo_operacion` tienen el mismo valor semántico escrito con distinto
casing (`DEPARTAMENTO`/`Departamento`/`departamento`, `ARRIENDO`/`Arriendo`/`arriendo`) —
normalizar en staging.

In [14]:
comuna_no_nula = df["comuna"].dropna().astype(str)
con_espacios = sorted(c for c in comuna_no_nula.unique() if c != c.strip())
print(f"Comunas con espacios al inicio/final: {len(con_espacios)} de {comuna_no_nula.nunique()} valores únicos")
print(con_espacios)
print("Comunas únicas tras strip():", comuna_no_nula.str.strip().nunique())

Comunas con espacios al inicio/final: 50 de 125 valores únicos
[' Cerrillos', ' Conchalí', ' Estación Central', ' Huechuraba', ' Independencia', ' La Cisterna', ' La Florida', ' La Reina', ' Las Condes', ' Lo Barnechea', ' Macul', ' Maipú', ' Peñalolén', ' Providencia', ' Pudahuel', ' Puente Alto', ' Quilicura', ' Quinta Normal', ' Recoleta', ' Renca', ' San Joaquín', ' San Miguel', ' Santiago', ' Vitacura', ' Ñuñoa', 'Cerrillos ', 'Conchalí ', 'Estación Central ', 'Huechuraba ', 'Independencia ', 'La Cisterna ', 'La Florida ', 'La Reina ', 'Las Condes ', 'Lo Barnechea ', 'Macul ', 'Maipú ', 'Peñalolén ', 'Providencia ', 'Pudahuel ', 'Puente Alto ', 'Quilicura ', 'Quinta Normal ', 'Recoleta ', 'Renca ', 'San Joaquín ', 'San Miguel ', 'Santiago ', 'Vitacura ', 'Ñuñoa ']
Comunas únicas tras strip(): 75


## 7. Volumen de registros por archivo mensual

In [15]:
df["archivo_origen"].value_counts().sort_index()

archivo_origen
2023-01.csv    4623
2023-02.csv    4719
2023-03.csv    5131
2023-04.csv    5044
2023-05.csv    5115
2023-06.csv    5347
2023-07.csv    5326
2023-08.csv    5458
2023-09.csv    4932
2023-10.csv    5429
2023-11.csv    5073
2023-12.csv    4766
2024-01.csv    5508
2024-02.csv    5221
2024-03.csv    5179
2024-04.csv    5114
2024-05.csv    4916
2024-06.csv    5037
2024-07.csv    4930
2024-08.csv    5482
2024-09.csv    5503
2024-10.csv    5506
2024-11.csv    5510
2024-12.csv    5505
Name: count, dtype: int64

## Resumen del perfilado y hallazgos

- **Esquema estable**: los 24 archivos comparten exactamente las mismas 12 columnas.
- **Nulos**: `comuna` y `contact_type` tienen nulos; el resto de columnas no.
- **Duplicados**: hay filas con `id` repetido; a validar en staging si corresponden al mismo
  aviso reprocesado por el scraper.
- **Precio en dos unidades**: `precio` mezcla UF y CLP según `divisa` — no es comparable sin
  normalizar a una sola unidad.
- **Casing inconsistente**: `tipo_propiedad` y `tipo_operacion` tienen el mismo valor en
  distintas variantes de mayúsculas/minúsculas.
- **Espacios en `comuna`**: varios valores tienen espacios al inicio/final que inflan el
  conteo de comunas únicas.
- **Fechas**: parsean sin error y `fecha_publicacion <= fecha_scraping` se cumple siempre en
  esta muestra.
- **Volumen por archivo**: varía entre ~4600 y ~5500 filas mensuales, sin caídas abruptas que
  sugieran una carga incompleta.

**Decisiones derivadas para staging**: normalizar `precio`/`divisa` a dos columnas de moneda comparables, unificar casing de `tipo_propiedad`/`tipo_operacion`, hacer `TRIM` + mapeo de `comuna`, y resolver los `id` duplicados. El detalle de outliers de precio y superficie (valores centinela, precio máximo, etc.) se investigó más a fondo durante la construcción de staging y queda documentado en `notebooks/03_validate_staging.ipynb`, no acá.